# 01 · Data audit & cleaning
**Question:** can we trust this data enough to base campaign decisions on it?

**Takeaways:** 41,188 contacts, 11.3% subscribed; 96% of customers had never been contacted before; `duration` is a post-call variable and is quarantined from targeting models.

In [1]:
import sys, json; sys.path.insert(0, '..')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import Image, display
from src import data, modelling, business, segmentation
from src.config import *
pd.set_option('display.width', 200); pd.set_option('display.float_format', '{:.3f}'.format)
df = data.clean(data.load_raw())

In [2]:
raw = data.load_raw()
print(raw.shape)
raw.isna().sum()[raw.isna().sum() > 0].to_frame('missing')

(41188, 28)


,missing
job,330
marital,80
education,1731
default,8597
housing,990
loan,990
age_group,5


**Missing values.** Missing job/education/default/housing/loan values are the bank's own 'unknown' answers, so they are kept as an explicit `unknown` level rather than imputed or dropped (default status is unknown for 21% of customers, so dropping would discard a fifth of the data).

In [3]:
print('Exact duplicate rows:', df.drop(columns='campaign_order').duplicated().sum(), '-> kept (could be distinct clients with identical attributes)')
print('Never previously contacted:', f"{1-df.was_contacted_before.mean():.1%}")
print('Rows with ages <=17:', (df.age<=17).sum())

Exact duplicate rows: 12 -> kept (could be distinct clients with identical attributes)
Never previously contacted: 96.3%
Rows with ages <=17: 5


**Feature engineering:** `age_group`, `campaign_intensity` (contacts in this campaign), `was_contacted_before` (from `pdays`=999 sentinel), `previous_success`, and `campaign_order` (row order is chronological, May 2008 → Nov 2010, used later for out-of-time validation).

**Leakage control:** `duration` (call length) is only known after the call ends, so it is used for description only.

In [4]:
df.groupby('subscribed').duration.describe()

,count,mean,std,min,25%,50%,75%,max
subscribed,,,,,,,,
0,36548.000,220.845,207.096,0.000,95.000,163.500,279.000,4918.000
1,4640.000,553.191,401.172,37.000,253.000,449.000,741.250,4199.000
